<div style="background:linear-gradient(135deg,#0f0c29,#302b63);padding:20px 24px;border-radius:10px;border-left:5px solid #4ade80;font-family:Arial,sans-serif;">
  <h2 style="margin:0;color:#4ade80;">🎮&nbsp;Tic Tac Toe — Minimax AI</h2>
  <p style="margin:8px 0 0;color:#bbb;font-size:14px;">Unbeatable AI via Minimax · 5 themes · variable board (3×3→6×6) · win animation · tkinter GUI</p>
</div>

## Overview

Fully featured Tic Tac Toe with two AI modes:
- **LightAI** — random moves (beatable)
- **MinimaxAI** — optimal play, provably unbeatable on 3×3

**Dependencies:** `tkinter`, `random`, `time` — all standard library.

> **Note:** This launches a desktop window. Execute the last cell in an environment that supports a display, or save as `tictactoe.py` and run from terminal.

## 1. Imports & Theme Configuration

In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
import random
import time

THEMES = {
    "Neon AI":            {"bg":"#1A1A40","btn_bg":"#6A0572","fg":"#00F5FF","x_color":"#FF00FF","o_color":"#00FFFF"},
    "Deep Learning Grid": {"bg":"#0F172A","btn_bg":"#1E293B","fg":"#38BDF8","x_color":"#FBC02D","o_color":"#BBDEFB"},
    "Quantum Glow":       {"bg":"#092635","btn_bg":"#1B4242","fg":"#9EC8B9","x_color":"#FFD700","o_color":"#76FF03"},
    "Neural Night":       {"bg":"#121212","btn_bg":"#212121","fg":"#26A69A","x_color":"#FF6F00","o_color":"#26C6DA"},
    "Data Science Wave":  {"bg":"#673AB7","btn_bg":"#FF9800","fg":"#E3F2FD","x_color":"#D84315","o_color":"#FFEA00"},
}

DIFFICULTY    = "LightAI"
board_size    = 3
board         = []
winning_line  = []
current_theme = "Neon AI"

## 2. AI Logic — Minimax Algorithm

Minimax exhaustively evaluates all future game states. O (AI) maximises the score, X (human) minimises. Score: `+1` = O wins, `-1` = X wins, `0` = draw.
On a standard 3×3 board the AI is mathematically unbeatable.

In [ ]:
def generate_winning_combos(size):
    combos = []
    for i in range(size):
        combos.append([i * size + j for j in range(size)])    # row
        combos.append([j * size + i for j in range(size)])    # col
    combos.append([i * (size + 1) for i in range(size)])      # main diagonal
    combos.append([i * (size - 1) for i in range(1, size+1)]) # anti-diagonal
    return combos

def get_winner():
    for combo in generate_winning_combos(board_size):
        vals = [board[i]["text"] for i in combo]
        if vals[0] != "" and vals.count(vals[0]) == board_size:
            return vals[0]
    return None

def light_ai():
    return random.choice([i for i, btn in enumerate(board) if btn["text"] == ""])

def minimax(is_maximizing):
    winner = get_winner()
    if winner == "O": return 1
    if winner == "X": return -1
    if all(btn["text"] != "" for btn in board): return 0
    best = -float("inf") if is_maximizing else float("inf")
    for i in range(len(board)):
        if board[i]["text"] == "":
            board[i]["text"] = "O" if is_maximizing else "X"
            score = minimax(not is_maximizing)
            board[i]["text"] = ""
            best = max(best, score) if is_maximizing else min(best, score)
    return best

def minimax_ai():
    best_score, best_move = -float("inf"), None
    for i in range(len(board)):
        if board[i]["text"] == "":
            board[i]["text"] = "O"
            score = minimax(False)
            board[i]["text"] = ""
            if score > best_score:
                best_score, best_move = score, i
    return best_move

def ai_move():
    empty = [i for i, btn in enumerate(board) if btn["text"] == ""]
    if not empty: return
    move = light_ai() if DIFFICULTY == "LightAI" else minimax_ai()
    root.after(500, lambda: make_move(move, "O"))

## 3. Game Logic — Win Detection & Board State

In [ ]:
def animate_win(combo):
    for _ in range(5):
        for i in combo:
            board[i].config(fg="red" if _ % 2 == 0 else "white")
        root.update()
        time.sleep(0.25)

def show_winner_message(winner):
    messages = {
        "X":   "🏆 Humanity Triumphs! X wins!",
        "O":   "🤖 AI Dominates! O wins!",
        "Tie": "⚖️ Perfect Balance — It's a Draw!",
    }
    messagebox.showinfo("Game Over", messages[winner])

def check_winner():
    global winning_line
    for combo in generate_winning_combos(board_size):
        vals = [board[i]["text"] for i in combo]
        if vals[0] != "" and vals.count(vals[0]) == board_size:
            winning_line = combo
            animate_win(winning_line)
            show_winner_message(vals[0])
            root.after(100, reset_game)
            return
    if all(btn["text"] != "" for btn in board):
        show_winner_message("Tie")
        root.after(100, reset_game)

def reset_game():
    for btn in board: btn.config(text="", fg="black")
    global winning_line; winning_line = []

def make_move(index, player):
    theme = THEMES[current_theme]
    if board[index]["text"] == "":
        board[index]["text"] = player
        board[index]["fg"]   = theme["x_color"] if player == "X" else theme["o_color"]
        check_winner()
        if player == "X" and not all(btn["text"] != "" for btn in board):
            ai_move()

## 4. GUI — Board, Controls & Theme Selector

In [ ]:
def draw_board():
    global board
    for w in frame.winfo_children(): w.destroy()
    board = []
    for i in range(board_size ** 2):
        btn = tk.Button(frame, text="", font=("Arial",14), width=5, height=2,
                        command=lambda i=i: make_move(i, "X"))
        btn.grid(row=i // board_size, column=i % board_size, padx=2, pady=2)
        board.append(btn)

def update_board_size(event):
    global board_size
    board_size = int(size_menu.get())
    draw_board()

def change_difficulty(event):
    global DIFFICULTY
    DIFFICULTY = diff_menu.get()

def apply_theme(theme_name):
    global current_theme
    if theme_name not in THEMES: return
    current_theme = theme_name
    t = THEMES[theme_name]
    root.configure(bg=t["bg"])
    frame.configure(bg=t["bg"])
    for lbl in (size_label, diff_label, theme_label):
        lbl.configure(bg=t["bg"], fg=t["fg"])
    restart_btn.configure(bg=t["btn_bg"])
    for btn in board: btn.configure(bg=t["bg"], fg="black")

## ▶ Launch Game Window

In [ ]:
root = tk.Tk()
root.title("Tic Tac Toe — AI Edition")
root.geometry("500x600")

frame = tk.Frame(root, bg="#222831")
frame.pack(pady=10)

size_label = tk.Label(root, text="Board Size:", font=("Arial",12), bg="#222831", fg="white")
size_label.pack()
size_menu  = ttk.Combobox(root, values=[3,4,5,6], state="readonly"); size_menu.set(3); size_menu.pack()
size_menu.bind("<<ComboboxSelected>>", update_board_size)

diff_label = tk.Label(root, text="AI Difficulty:", font=("Arial",12), bg="#222831", fg="white")
diff_label.pack()
diff_menu  = ttk.Combobox(root, values=["LightAI","MinimaxAI"], state="readonly"); diff_menu.set("LightAI"); diff_menu.pack()
diff_menu.bind("<<ComboboxSelected>>", change_difficulty)

theme_label = tk.Label(root, text="Theme:", font=("Arial",12), bg="#222831", fg="white")
theme_label.pack()
theme_menu  = ttk.Combobox(root, values=list(THEMES.keys()), state="readonly"); theme_menu.set("Neon AI"); theme_menu.pack()
theme_menu.bind("<<ComboboxSelected>>", lambda e: apply_theme(theme_menu.get()))

restart_btn = tk.Button(root, text="🔄 Restart", command=reset_game,
                        font=("Arial",14,"bold"), bg="#00cc99", fg="white")
restart_btn.pack(pady=10)

draw_board()
root.mainloop()